# Fitting a joint RV + astrometric orbit, the open way

This tutorial is the **joint sibling** of `fit_rv_orbit.ipynb` and
`fit_astrometric_orbit.ipynb`.  It fits the *same* synthetic SB1 orbit to
**both** channels at once -- the radial velocities and the Gaia along-scan
epoch astrometry -- in the spirit of the
[emcee line-fitting tutorial](https://emcee.readthedocs.io/en/stable/tutorials/line/):
everything is explicit, editable, and close to the data.  We progress from
two independent period searches, to per-channel closed-form baselines, to a
quick joint maximum-likelihood point, to a hand-written joint probability
and a bare `emcee` run.

The two channels share the **non-linear orbit shape** `(P, e, tau)`: the
period, eccentricity and periastron phase are the *same* physical orbit
seen two ways.  Each channel keeps its own *linear* amplitudes, solved in
closed form **inside** the objective:

- RV is linear in `(gamma, K cos omega, K sin omega)` (audited
  `linear_solve_rv`);
- the along-scan model is linear in nine Thiele-Innes + astrometric
  amplitudes `(A, B, F, G, ra_offset, dec_offset, pmra, pmdec, plx)`
  (audited `linear_solve_ti`).

So we sample only the three shared shape parameters and let both linear
solves run inside the likelihood -- exactly the trick the two sibling
notebooks use, now sharing one shape.  This keeps the sampler in three
dimensions and sidesteps the high-dimensional basin-trap that bites the
full engine fit.

We never touch the production sampler's internal latent reparametrization.
We only *call* audited, public `orblet` functions for the forward models,
the two linear solves (whose closed-form marginal evidences are the joint
likelihood we sample) and the Thiele-Innes -> Campbell inversion, plus
standard `numpy`, `scipy`, `emcee`, and `corner`.

**Shared-epoch convention (load-bearing).** Both channels use the **same**
reference epoch `EPOCH_REF` (MJD) and the **same** periastron phase `tau`,
with `tp = tau * P + EPOCH_REF` in both the RV model and the along-scan
model.  This is what makes `tau` a *shared* parameter: a single periastron
time ties the velocity curve and the photocenter track together.  We put
both time axes on MJD (adding the J2010 anchor `MJD_J2010_TCB` to the
bundle's day-from-J2010 times) so the two channels share one clock.

**Units and conventions** (reused from the audited model, not re-derived):

- `P` orbital period in **days**; internally converted to Keplerian years
  (`P_yr = P / 365.25`) at the model interface.
- `e` eccentricity, dimensionless, `0 <= e < 1`.
- `tau` periastron phase fraction in `[0, 1)`; `tp = tau * P + EPOCH_REF`.
- RV: `K_kms` km/s primary semi-amplitude, `omega` rad (primary frame),
  `gamma` km/s systemic offset.  RV fixes `sin i = 1`, so `K_kms` alone
  measures the SB1 mass function, not the true companion mass.
- Astrometry: `A, B, F, G` photocenter Thiele-Innes amplitudes in **mas**;
  derived Campbell `i` in `[0, pi]` (full sphere, Gaia convention),
  `Omega`, `omega` rad, `a_phot` mas.
- Along-scan projection (audited `along_scan_model`):
  `model = d_ra*sin(psi) + d_dec*cos(psi) + 5-parameter astrometry`, with
  `psi` the scan angle, `pmra = mu_alpha*` already cos-delta-corrected, and
  the parallax term added as `plx * parallax_factor_al`.

The astrometry **measures** the inclination `i` (the photocenter track is
two-dimensional); the optional appendix uses that *measured* `sin i` with
the RV mass function to quote a **candidate** companion mass.

> **Which fit is this?** This is the simple **quick-look / linearized
> baseline**: two closed-form linear solves at a shared fixed orbital
> shape, a `scipy` maximum-likelihood refinement of the shared shape, and
> an optional bare `emcee` run, all written out in the open.  §8 and §9
> climb from there to the shared-angle and the all-parameter joint fits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.optimize import minimize
from astropy.timeseries import LombScargle
import emcee
import corner

# Audited, public building blocks. We use ONLY the forward models and the
# two linear solves (whose closed-form marginal evidences ARE the joint
# likelihood, section 5) plus the Thiele-Innes -> Campbell inversion --
# NOT the production samplers -- so the joint probability we sample is
# written out explicitly below.
from orblet.simulate.bundles import load_simulated_inputs
from orblet import (
    semi_amplitude_kms,
    rv_design_matrix,
    linear_solve_rv,
    recover_K,
    recover_omega,
    linear_solve_ti,
    ti_to_kepler,
    scan_ti_frequency,
    ti_design_matrix,
)
from orblet.model import (
    rv_model,
    along_scan_model,
    thiele_innes_xy,
    kepler_xy_orbit,
    tp_from_disk_angle,
)
from orblet.interpret.companion_mass import companion_mass_from_rv_posterior
from orblet.interpret.flux_ratio import MeasuredSinI
from orblet.constants import (
    MJD_J2010_TCB, DAYS_PER_KEPLER_YEAR, G_SI, MSUN_KG,
)

rng = np.random.default_rng(0)

## 1. The data

We load the synthetic demo bundle (the toy orbit, strongly detected in both channels) and pull
out **both** the radial velocities and the epoch astrometry.  The bundle is
fully in-memory and synthetic -- no files are read, no real targets are
touched.

We put both time axes on **MJD** by adding the J2010 anchor
`MJD_J2010_TCB`, and adopt the bundle's own reference epoch
`EPOCH_REF = t_ref_mjd` for **both** channels.  This shared epoch sets the
common periastron-time and proper-motion zero points -- the heart of the
joint fit.

In [ ]:
bundle = load_simulated_inputs(seed=0)  # the toy orbit on the demo cadence

# --- RV channel ---
rv = bundle.rv_data
valid = np.asarray(rv['rv_validity_flag'])
t_rv = np.asarray(rv['obs_time_rv'], dtype=float)[valid] + MJD_J2010_TCB  # MJD
v = np.asarray(rv['radial_velocity'], dtype=float)[valid]                 # km/s
verr = np.asarray(rv['radial_velocity_err'], dtype=float)[valid]          # km/s

# --- Astrometry channel ---
ad = bundle.astro_data
t_ast = np.asarray(ad['obs_time'], dtype=float) + MJD_J2010_TCB  # MJD (TCB)
psi = np.asarray(ad['scan_angle'], dtype=float)                  # rad
pf = np.asarray(ad['parallax_factor_al'], dtype=float)           # dimensionless
d_obs = np.asarray(ad['centroid_pos'], dtype=float)              # mas (along scan)
sigma = np.asarray(ad['centroid_pos_err'], dtype=float)          # mas

# Shared reference epoch for BOTH channels (the joint anchor).
# EPOCH_REF is the position/proper-motion reference epoch (J2016), NOT the
# periastron epoch; tau is a free phase whose absolute zero-point is
# immaterial as long as BOTH channels share the same EPOCH_REF.
EPOCH_REF = float(bundle.truth.t_ref_mjd)  # MJD

# Total system mass: an input to the audited K<->mass map. The astrometry
# constrains the geometry, not the mass split; this is editable.
M_TOTAL_MSUN = 10.5

print(f'{t_rv.size} RV epochs, {t_ast.size} astrometric transits')
print(f'shared reference epoch EPOCH_REF = {EPOCH_REF:.1f} MJD')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.errorbar(t_rv - EPOCH_REF, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2)
ax1.set_xlabel('time - EPOCH_REF (days)')
ax1.set_ylabel('radial velocity (km/s)')
ax1.set_title('RV channel')

ax2.errorbar(t_ast - EPOCH_REF, d_obs, yerr=sigma, fmt='o', color='k',
             ms=4, capsize=2)
ax2.set_xlabel('time - EPOCH_REF (days)')
ax2.set_ylabel('along-scan abscissa (mas)')
ax2.set_title('Astrometry channel (along-scan)')
plt.tight_layout()
plt.show()

## 2. Period search in both channels

A joint fit only makes sense if both channels see the **same** period.  We
run a Lomb-Scargle periodogram on the RVs and the audited
`scan_ti_frequency` Thiele-Innes scan on the astrometry, then **check**
the two agree before fixing a shared seed. The check below is a coarse
gate on two point estimates from grids of different resolution, not a
σ-level test; the joint posterior in §6 is the real comparison.

A caveat carried from the astrometric sibling: the Gaia scanning law makes
the true period (~185 d here) sit near the **6-month parallax alias**, so
the top TI peak is itself alias-flagged.  The flag is a review warning, not
a disqualifier -- and the RV periodogram, which has no parallax, is the
independent check that the period is real.

In [ ]:
# RV: Lomb-Scargle.
ls = LombScargle(t_rv, v, verr)
freq, power = ls.autopower(minimum_frequency=1.0 / 500.0,
                           maximum_frequency=1.0 / 50.0,
                           samples_per_peak=20)
periods = 1.0 / freq
P_rv = float(periods[np.argmax(power)])

# Astrometry: Thiele-Innes frequency scan.
peaks = scan_ti_frequency(
    t_ast, psi, pf, d_obs, sigma,
    f_min_per_day=1.0 / 500.0, f_max_per_day=1.0 / 50.0, oversample=4.0,
    ecc_grid=[0.0, 0.2, 0.4, 0.6],
    tau_grid=np.linspace(0.0, 1.0, 8, endpoint=False),
    epoch_ref_mjd=EPOCH_REF, top_k=6,
)
P_ast = float(peaks[0].P_days)

# Shared period seed: average the two -- AFTER checking they agree.
P_seed = 0.5 * (P_rv + P_ast)
print(f'RV Lomb-Scargle period : {P_rv:.2f} d')
print(f'astrometry TI period    : {P_ast:.2f} d (alias-flagged={peaks[0].flagged})')
print(f'shared period seed      : {P_seed:.2f} d')

# The agreement is CHECKED, not assumed. Both numbers are grid-quantised
# point estimates (periodogram peak; TI scan node), so this is a coarse
# 5% gate rather than a sigma-level test.
rel_diff = abs(P_rv - P_ast) / (0.5 * (P_rv + P_ast))
print(f'relative difference     : {rel_diff:.2%}')
if rel_diff > 0.05:
    print('  -> the channels do NOT agree at the 5% level: do not average '
          'them.\n     Diagnose each channel on its own before any joint fit.')
else:
    print('  -> agree at the 5% level; averaging them is a reasonable seed.')

In [ ]:
P_six_month = 365.25 / 2.0
P_one_year = 365.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(periods, power, color='k', lw=1)
ax1.axvline(P_rv, color='C0', ls='--', label=f'RV peak = {P_rv:.1f} d')
ax1.set_xlabel('period (days)')
ax1.set_ylabel('Lomb-Scargle power')
ax1.set_title('RV periodogram')
ax1.legend(fontsize=8)

P_peaks = np.array([p.P_days for p in peaks])
L_peaks = np.array([p.logL_marginal for p in peaks])
flagged = np.array([p.flagged for p in peaks])
ax2.scatter(P_peaks[~flagged], L_peaks[~flagged], color='k', zorder=5,
            label='TI peaks')
if flagged.any():
    ax2.scatter(P_peaks[flagged], L_peaks[flagged], color='C3', marker='x',
                s=60, zorder=6, label='alias-flagged')
for P_alias, lab in [(P_one_year, '1 yr'), (P_six_month, '6 mo')]:
    ax2.axvline(P_alias, color='C3', ls='--', alpha=0.6)
    ax2.text(P_alias, ax2.get_ylim()[1], f' {lab} alias', color='C3',
             va='top', ha='left', fontsize=8)
ax2.axvline(P_ast, color='C0', ls=':', label=f'TI peak = {P_ast:.1f} d')
ax2.set_xlabel('period (days)')
ax2.set_ylabel('logL_marginal')
ax2.set_title('Astrometry TI scan')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Closed-form baselines at the shared shape

At a **fixed** shared shape `(P, e, tau)`, each channel is *linear* in its
own amplitudes.  We run both closed-form solves at the shared period seed
(scanning `tau` for each, with a guessed eccentricity) and read off the
quick per-channel fits:

- RV `linear_solve_rv` -> `(K, omega, gamma)` and the mass function;
- astrometry `linear_solve_ti` -> `(A, B, F, G, ...)`, then `ti_to_kepler`
  -> Campbell `(i, Omega, omega, a_phot)`.

These are the same two baselines as in the sibling notebooks, now read at
one shared period.

In [ ]:
def solve_rv(P_days, e, tau):
    """Closed-form RV amplitudes (K, omega, gamma) at fixed shape."""
    X = rv_design_matrix(t_rv, period_yr=P_days / DAYS_PER_KEPLER_YEAR, ecc=e,
                         tau=tau, epoch_ref_mjd=EPOCH_REF)
    sol = linear_solve_rv(v, verr, X)
    K = recover_K(sol.beta)
    omega = recover_omega(sol.beta) % (2.0 * np.pi)
    gamma = float(sol.beta[0])
    return sol, K, omega, gamma


def solve_ti(P_days, e, tau):
    """Closed-form 9 along-scan amplitudes at fixed shape."""
    X = ti_design_matrix(t_ast, psi, pf, f_per_day=1.0 / P_days, ecc=e,
                         tau=tau, epoch_ref_mjd=EPOCH_REF)
    return linear_solve_ti(d_obs, sigma, X)


def campbell_from_beta(beta):
    """Thiele-Innes (A,B,F,G)+plx -> Campbell (i, Omega, omega, a_phot)."""
    kep = ti_to_kepler(
        {'A_mas': beta[0:1], 'B_mas': beta[1:2], 'F_mas': beta[2:3],
         'G_mas': beta[3:4], 'plx_mas': beta[8:9]},
        plx_key='plx_mas',
    )
    return {
        'i_deg': float(np.rad2deg(kep['inc_rad'][0])),
        'Omega_deg': float(np.rad2deg(kep['Omega_rad'][0])),
        'omega_deg': float(np.rad2deg(kep['omega_rad'][0])),
        'a_phot_mas': float(kep['a_phot_mas'][0]),
    }

In [ ]:
e_guess = 0.4
tau_grid = np.linspace(0.0, 1.0, 200, endpoint=False)

# RV: scan tau by chi2.
chi2_rv = np.array([solve_rv(P_seed, e_guess, tau)[0].chi2 for tau in tau_grid])
tau_rv = float(tau_grid[np.argmin(chi2_rv)])
sol_rv, K_b, omega_b, gamma_b = solve_rv(P_seed, e_guess, tau_rv)

# Astrometry: scan tau by chi2.
chi2_ti = np.array([solve_ti(P_seed, e_guess, tau).chi2 for tau in tau_grid])
tau_ti = float(tau_grid[np.argmin(chi2_ti)])
sol_ti = solve_ti(P_seed, e_guess, tau_ti)
camp_b = campbell_from_beta(sol_ti.beta)

print(f'RV baseline  (tau={tau_rv:.3f}): K={K_b:.2f} km/s, '
      f'omega={omega_b:.3f} rad, gamma={gamma_b:.3f} km/s')
print(f'TI baseline  (tau={tau_ti:.3f}): i={camp_b["i_deg"]:.1f} deg, '
      f'Omega={camp_b["Omega_deg"]:.1f} deg, omega={camp_b["omega_deg"]:.1f} deg, '
      f'a_phot={camp_b["a_phot_mas"]:.3f} mas')

## 4. Quick joint maximum-likelihood point

Now we optimise the **shared** non-linear shape `(P, e, tau)` with
`scipy.optimize.minimize`, solving *both* channels' linear amplitudes in
closed form inside the objective.  The objective is the negative **sum** of
the two audited marginal log-likelihood ranking scores -- one shape, two
channels, added together.  We try a few starts.

In [ ]:
def neg_joint_marginal(shape):
    P, e, tau = shape
    if not (50.0 < P < 500.0 and 0.0 <= e < 0.9 and 0.0 <= tau < 1.0):
        return 1e12
    try:
        sol_r = solve_rv(P, e, tau)[0]
        sol_t = solve_ti(P, e, tau)
    except Exception:
        return 1e12
    # Joint marginal score = sum of the two audited per-channel scores.
    return -(sol_r.logL_marginal + sol_t.logL_marginal)


best = None
for e0 in (0.1, 0.4, 0.6):
    for tau0 in np.linspace(0.05, 0.95, 5):
        res = minimize(neg_joint_marginal, [P_seed, e0, tau0],
                       method='Nelder-Mead')
        if best is None or res.fun < best.fun:
            best = res

P_ml, e_ml, tau_ml = best.x
sol_rv_ml, K_ml, omega_ml, gamma_ml = solve_rv(P_ml, e_ml, tau_ml)
camp_ml = campbell_from_beta(solve_ti(P_ml, e_ml, tau_ml).beta)
theta_ml = np.array([P_ml, e_ml, tau_ml])
print('quick joint ML point (shared shape):')
print(f'  P   = {P_ml:.2f} d')
print(f'  e   = {e_ml:.3f}')
print(f'  tau = {tau_ml:.3f}')
print('RV amplitudes:    K=%.2f km/s, omega=%.3f rad, gamma=%.3f km/s'
      % (K_ml, omega_ml, gamma_ml))
print('astrometry Campbell: i=%.1f deg, a_phot=%.3f mas'
      % (camp_ml['i_deg'], camp_ml['a_phot_mas']))

## 5b. Optional: a soft omega-consistency penalty (default OFF)

The baseline joint fit above ties the two channels only through the
**shared shape** `(P, e, tau)`; each channel recovers its own `omega`
independently, and section 7 *co-validates* by checking the two agree.
That co-validation is the honest, separate-fit-plus-closure picture.

As an **optional** experiment we can instead encode "same orbit" as a
soft **prior** on the two arguments of periastron. At a fixed shape we
already solve both channels in closed form, so both `omega` come for
free:

- `omega_RV = atan2(S, C)` from the RV solve, with analytic 1-sigma
  `sigma_RV` propagated from `sol_rv.cov` using
  `d(omega)/dC = -S / (C^2 + S^2)`, `d(omega)/dS = C / (C^2 + S^2)`;
- `omega_ast` from the TI solve via `ti_to_kepler`, with `sigma_ast`
  from a **deterministic finite-difference Jacobian** `J = d(omega)/d(beta_ti)`
  propagated through `sol_ti.cov` as `J Sigma J^T` (NOT per-step
  Monte-Carlo, so the likelihood stays smooth and reproducible).

The penalty is a Gaussian log-prior on the wrapped difference, and it is
added in `log_prior` — **not** in the likelihood. It encodes a modelling
belief ("one orbit"); it does not describe the data:

$$ \Delta = \mathrm{wrap}(\omega_{RV} - \omega_{ast}), \qquad
   \log p = -\tfrac{1}{2}\,\frac{\Delta^2}{\sigma_{RV}^2 + \sigma_{ast}^2}. $$

**Four caveats.**

1. **`omega` is circular, and we fold the TI branch degeneracy.** We
   compare with the *wrapped* difference `atan2(sin Delta, cos Delta)` in
   `(-pi, pi]`, never a raw subtraction, so `omega ~ 0` and `omega ~ 2pi`
   are treated as identical. The Thiele-Innes inversion reports `omega`
   on one of two degenerate branches (`omega` vs `omega + pi`, paired
   with `Omega` vs `Omega + pi`) that give identical astrometry, so the
   penalty compares to the *nearer* of `omega_ast` and `omega_ast + pi`.
   This ties the two orbits up to that intrinsic astrometric ambiguity,
   which the RV then helps resolve.
2. **This is a soft PRIOR, not a rigorous shared-`omega` parameter.** It
   nudges the two channels toward a common `omega` but does not enforce a
   single shared latent. The rigorous version is a full Campbell / joint
   fit with one shared `omega` -- a future, deeper layer.
3. **Both `omega` are in the PRIMARY frame.** The RV model and
   `ti_to_kepler` both report primary-frame `omega`, so the two are
   directly comparable (no frame flip needed).
4. **Its width is DATA-DERIVED, so the term is not an independent
   constraint.** `sigma_RV` and `sigma_ast` are propagated from *this*
   dataset's own amplitude covariances, and the same data then enter the
   likelihood: switching the term on double-counts that information and
   tightens the posterior by an amount nothing external justifies. Treat
   it as a regulariser for exploration, and keep it **OFF** for anything
   you intend to quote.

`USE_OMEGA_PENALTY` gates the term inside `log_prior`. It defaults to
**`False`** -- the baseline stays separate-fit + closure (toggling it off
recovers the honest co-validation). Set it to `True` to switch on the
soft tie.

In [ ]:
# Default OFF: the baseline joint fit stays separate-fit + closure.
# Toggle to True to add the soft omega-consistency tie to the likelihood.
USE_OMEGA_PENALTY = False


def log_consistency_penalty(P_days, e, tau):
    """Soft Gaussian log-prior tying omega_RV to omega_ast (primary frame).

    Returned from log_prior (NOT the likelihood): it encodes "one
    orbit", it does not describe the data. Its width comes from THIS
    dataset's own covariances, so it is a regulariser, not an
    independent constraint -- see the caveats above.

    Returns -0.5 * Delta^2 / (sigma_RV^2 + sigma_ast^2), where Delta is the
    WRAPPED difference of the two arguments of periastron recovered in
    closed form at the shared shape. sigma_RV is propagated analytically
    from the RV amplitude covariance; sigma_ast from a deterministic
    finite-difference Jacobian of omega(beta_ti) through the TI covariance.
    """
    # --- RV side: omega_RV and its analytic sigma from (C, S) ---------------
    sol_rv, _, _, _ = solve_rv(P_days, e, tau)
    C, S = sol_rv.beta[1], sol_rv.beta[2]          # C = K cos w, S = K sin w
    omega_rv = np.arctan2(S, C)
    denom = C * C + S * S                          # = K^2
    # omega = atan2(S, C): d w/dC = -S/(C^2+S^2), d w/dS = C/(C^2+S^2).
    g = np.array([0.0, -S / denom, C / denom])     # grad over (gamma, C, S)
    var_rv = float(g @ sol_rv.cov @ g)

    # --- Astrometry side: omega_ast and its FD-Jacobian sigma --------------
    sol_ti = solve_ti(P_days, e, tau)
    beta = sol_ti.beta

    def omega_ast_of(beta_vec):
        """Primary-frame omega (rad) from a TI amplitude vector."""
        kep = ti_to_kepler(
            {'A_mas': beta_vec[0:1], 'B_mas': beta_vec[1:2], 'F_mas': beta_vec[2:3],
             'G_mas': beta_vec[3:4], 'plx_mas': beta_vec[8:9]}, plx_key='plx_mas')
        return float(kep['omega_rad'][0])

    omega_ast = omega_ast_of(beta)
    # Deterministic central finite-difference Jacobian d(omega)/d(beta_ti).
    # Three deliberate choices:
    #  - range(4): omega is a function of (A, B, F, G) ONLY, so the five
    #    astrometric-nuisance partials are exactly zero -- computing them
    #    is wasted work (and pure round-off);
    #  - each perturbed omega is evaluated ONCE and reused in sin and cos
    #    (the old form called the inverter four times per component);
    #  - a RELATIVE step (floored at 1) so it stays meaningful whatever
    #    the mas-scale of the amplitudes.
    # The difference is wrapped so the derivative survives the atan2 cut.
    J = np.zeros(beta.size)
    for k in range(4):
        h = 1e-6 * max(abs(float(beta[k])), 1.0)
        bp, bm = beta.copy(), beta.copy()
        bp[k] += h
        bm[k] -= h
        w_plus, w_minus = omega_ast_of(bp), omega_ast_of(bm)
        dphi = np.arctan2(np.sin(w_plus - w_minus),
                          np.cos(w_plus - w_minus))
        J[k] = dphi / (2.0 * h)
    var_ast = float(J @ sol_ti.cov @ J)

    # --- wrapped difference, folding the TI (omega, Omega) 180-deg branch --
    # ti_to_kepler reports omega on one of two degenerate branches
    # (omega vs omega+pi, paired with Omega vs Omega+pi); both give
    # identical astrometry. Compare omega_rv to BOTH branches and keep the
    # nearer wrapped difference, so a physically-consistent orbit on the
    # flipped branch is not wrongly penalized. The RV then breaks the
    # astrometric branch ambiguity (a feature, not a bug).
    d0 = np.arctan2(np.sin(omega_rv - omega_ast),
                    np.cos(omega_rv - omega_ast))
    d1 = np.arctan2(np.sin(omega_rv - (omega_ast + np.pi)),
                    np.cos(omega_rv - (omega_ast + np.pi)))
    delta = d0 if abs(d0) <= abs(d1) else d1
    return -0.5 * delta ** 2 / (var_rv + var_ast)


# Quick look (informational): the penalty value at the quick ML shape.
print('omega-consistency penalty at the ML shape:',
      log_consistency_penalty(*theta_ml),
      '(USE_OMEGA_PENALTY =', USE_OMEGA_PENALTY, ')')

## 5. An explicit, editable joint probability

Following the emcee tutorial, we write the joint probability out by hand so
it is easy to read and to change.  The joint log-likelihood is the **sum of
the two channels' MARGINAL log-evidences** at the **shared** `(P, e, tau)` --
no new orbit math:

- the RV term is `linear_solve_rv(...).logL_marginal`, with
  `(gamma, K cos w, K sin w)` integrated out in closed form;
- the astrometry term is `linear_solve_ti(...).logL_marginal`, with the
  nine along-scan amplitudes integrated out the same way.

That summed score is **exactly what section 4 optimised**, so the sampler
and the quick ML point target the same function. Scoring each channel's
Gaussian *at* its fitted amplitudes instead would be the **profile**
likelihood -- the amplitudes' uncertainty dropped rather than integrated,
and over-confident. (`fit_rv_orbit.ipynb` §5 makes the same choice.)

Each channel's linear amplitudes are therefore integrated out **inside** the
likelihood, and we sample only the three shared shape parameters.  **Both
channels use the same `EPOCH_REF` and the same `tau` zero-point**, so the
periastron time `tp = tau * P + EPOCH_REF` is identical in the velocity
curve and the photocenter track -- that shared `tp` is what couples the two
channels.  The prior is a simple editable box on the shared shape.

In [ ]:
# Box prior bounds on the shared non-linear shape theta = (P, e, tau).
P_LO, P_HI = 150.0, 250.0  # days


def rv_marginal_logL(P_days, e, tau):
    """RV channel MARGINAL log-evidence at the shared shape.

    The three linear amplitudes (gamma, K cos w, K sin w) are integrated
    out in closed form by the audited linear_solve_rv; the returned
    sol.logL_marginal is the same flat-prior score section 4 optimised.
    """
    return float(solve_rv(P_days, e, tau)[0].logL_marginal)


def astro_along_scan_at(P_days, e, tau):
    """Along-scan model + beta at the shared shape (9 amplitudes inside).

    Used for the RESIDUAL plots in section 7. The likelihood below does
    not need the model itself -- it uses the marginal evidence.
    """
    sol = solve_ti(P_days, e, tau)
    beta = sol.beta
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    # Shared periastron time: tp = tau*P + EPOCH_REF, the SAME convention
    # the RV model uses internally -> tau is a genuinely shared parameter.
    disk_x, disk_y = np.cos(2 * np.pi * tau), np.sin(2 * np.pi * tau)
    tp = tp_from_disk_angle(disk_x, disk_y, period_days=P_days,
                            epoch_ref_mjd=EPOCH_REF)
    d_ra, d_dec = thiele_innes_xy(t_ast, period_yr=P_yr, ecc=e,
                                  A_mas=beta[0], B_mas=beta[1],
                                  F_mas=beta[2], G_mas=beta[3], tp_mjd=tp)
    model = along_scan_model(
        d_ra=d_ra, d_dec=d_dec, psi=psi, t_mjd=t_ast, epoch_ref_mjd=EPOCH_REF,
        ra_offset_mas=beta[4], dec_offset_mas=beta[5],
        pmra_masyr=beta[6], pmdec_masyr=beta[7], plx_mas=beta[8],
        parallax_factor_al=pf,
    )
    return model, beta


def astro_marginal_logL(P_days, e, tau):
    """Astrometry channel MARGINAL log-evidence at the shared shape.

    The nine along-scan amplitudes are integrated out in closed form by
    the audited linear_solve_ti.
    """
    return float(solve_ti(P_days, e, tau).logL_marginal)


def log_prior(theta):
    """Box prior on the shared shape (+ the optional omega tie).

    The omega-consistency term lives HERE, not in the likelihood: it is a
    modelling belief ("one orbit"), not a statement about the data. It
    is OFF by default (section 5b).
    """
    P, e, tau = theta
    if not (P_LO < P < P_HI and 0.0 <= e < 0.9 and 0.0 <= tau < 1.0):
        return -np.inf
    if USE_OMEGA_PENALTY:
        return log_consistency_penalty(P, e, tau)
    return 0.0


def log_likelihood_joint(theta):
    """Joint MARGINAL log-L: RV evidence + astrometry evidence at the
    SHARED (P, e, tau) -- the same summed score section 4 optimised.

    Scoring each channel's Gaussian AT its fitted amplitudes instead
    would be the profile likelihood: over-confident, and a different
    target from the quick ML point. The optional omega tie is a PRIOR
    and lives in log_prior above.
    """
    P, e, tau = theta
    try:
        return rv_marginal_logL(P, e, tau) + astro_marginal_logL(P, e, tau)
    except Exception:
        return -np.inf


def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ll = log_likelihood_joint(theta)
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


print('joint log_probability at the quick ML point:', log_probability(theta_ml))

## 6. Sample the joint posterior with emcee

A bare `emcee.EnsembleSampler` seeded in a tiny Gaussian ball around the
quick joint ML point.  We sample only the three **shared** shape
parameters, run a modest chain, discard a burn-in, and look at the traces
and a corner plot.

In [ ]:
n_walkers = 24
n_dim = 3
n_steps = 1500

p0 = theta_ml + 1e-3 * rng.standard_normal((n_walkers, n_dim))
p0[:, 1] = np.clip(p0[:, 1], 0.0, 0.89)   # keep e in support
p0[:, 2] = np.clip(p0[:, 2], 0.0, 0.999)  # keep tau in [0, 1)

sampler = emcee.EnsembleSampler(n_walkers, n_dim, log_probability)
sampler.run_mcmc(p0, n_steps, progress=True)
print('mean acceptance fraction:', np.mean(sampler.acceptance_fraction))

In [ ]:
labels = ['P (d)', 'e', 'tau']
fig, axes = plt.subplots(n_dim, figsize=(8, 5), sharex=True)
chain = sampler.get_chain()
for i in range(n_dim):
    axes[i].plot(chain[:, :, i], color='k', alpha=0.3, lw=0.5)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel('step')
plt.show()

In [ ]:
burnin = 500
thin = 5
flat = sampler.get_chain(discard=burnin, thin=thin, flat=True)
fig = corner.corner(flat, labels=labels, show_titles=True,
                    title_fmt='.3f', quantiles=[0.16, 0.5, 0.84])
plt.show()

## 7. Co-validation and a look at the data

The joint fit is only trustworthy if it **agrees** with the two independent
single-channel fits.  Co-validation here is a **consistency check**, not a
new measurement: we confirm the shared `(P, e, tau)` is consistent with
what each channel says on its own, and that the RV-only and astrometry-only
amplitudes describe the *same* orbit.

We check three things:

1. the joint `(P, e)` posterior overlaps the per-channel period/eccentricity
   estimates;
2. the projected semi-major axes: `a_spec = a1 sin i` from RV
   (`= K P sqrt(1-e^2) / 2pi`) versus `a_phot sin i` from astrometry.
   Under a **dark** companion (`beta = 0`) the photocentre traces the
   star's *full* reflex orbit, so `a_phot = a1` **exactly** -- there is no
   mass factor between them. Their fractional difference is the
   photocentre **deficit** (manual §4.3, all axes in AU)

   `D = 1 - a_phot sin i / a_spec`,  and exactly `D = beta / (B (1 + beta))`

   (`beta` = companion light fraction, `B` = its mass fraction), so
   `D ~ beta/B` when `beta << 1`. We report `D` with a credible interval;
3. the data themselves: a phase-folded RV curve and a sky-plane photocenter
   orbit, each overlaid with the joint posterior model and its residuals.

In [ ]:
# Per-sample physical quantities pushed through the closed-form solves.
# At a fixed shape each channel's amplitudes have the conditional Gaussian
# posterior N(sol.beta, sol.cov), so we DRAW from it per sample instead of
# taking the point estimate sol.beta. Drawing carries the amplitudes'
# proper conditional uncertainty into (K, omega, i, a_phot) -- and into
# the candidate mass in the appendix; a point estimate would make all of
# them artificially sharp (under-dispersed). Same recipe as
# fit_rv_orbit.ipynb section 5.
Ks, omegas, gammas = [], [], []
betas = []
for (P, e, tau) in flat:
    sol_r = solve_rv(P, e, tau)[0]
    beta_rv = rng.multivariate_normal(sol_r.beta, sol_r.cov)
    Ks.append(recover_K(beta_rv))
    omegas.append(recover_omega(beta_rv) % (2.0 * np.pi))
    gammas.append(float(beta_rv[0]))
    sol_t = solve_ti(P, e, tau)
    betas.append(rng.multivariate_normal(sol_t.beta, sol_t.cov))
Ks = np.array(Ks); omegas = np.array(omegas); gammas = np.array(gammas)
betas = np.array(betas)

kep = ti_to_kepler(
    {'A_mas': betas[:, 0], 'B_mas': betas[:, 1], 'F_mas': betas[:, 2],
     'G_mas': betas[:, 3], 'plx_mas': betas[:, 8]}, plx_key='plx_mas')
i_deg = np.rad2deg(kep['inc_rad'])
a_phot_mas = kep['a_phot_mas']
sin_i = np.sin(kep['inc_rad'])

Ps, es, taus = flat[:, 0], flat[:, 1], flat[:, 2]

# a1 sin i [AU] from RV: K [km/s] * P [days] * sqrt(1-e^2) / 2pi (audited
# reduction, see orblet.a1_sini_from_rv_chain). _AU_PER_KMS_DAY
# converts (km/s * day) -> AU.
_AU_PER_KMS_DAY = 86400.0 / 149597870.7
a1_sini_au = Ks * (Ps * np.sqrt(1.0 - es ** 2)) / (2.0 * np.pi) * _AU_PER_KMS_DAY
# a_phot sin i [AU] from astrometry. ti_to_kepler already divided by the
# FITTED per-draw parallax column we handed it (betas[:, 8]), so a_phot_au
# is the fitted axis -- dividing by bundle.truth.parallax_mas instead
# would leak the answer into a check meant to test it.
a_phot_au = kep['a_phot_au']
a_phot_sini_au = a_phot_au * sin_i

truth = bundle.truth
report = [
    ('P (d)',   Ps, truth.P_days),
    ('e',       es, truth.e),
    ('K (km/s)', Ks, truth.K1_kms),
    ('i (deg)',  i_deg, np.rad2deg(truth.i_rad)),
    ('a_phot (mas)', a_phot_mas, truth.a_phot_mas),
]
print('parameter         16%       50%       84%      truth')
pct = {}
for name, arr, tv in report:
    q = np.percentile(arr, [16, 50, 84])
    pct[name] = q
    print(f'{name:14s} {q[0]:9.3f} {q[1]:9.3f} {q[2]:9.3f}  {tv:9.3f}')

In [ ]:
# CONSISTENCY check 1: joint (P,e) vs independent single-channel estimates.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.hist(Ps, bins=40, color='0.7', density=True)
ax1.axvline(P_rv, color='C0', ls='--', label=f'RV-only P = {P_rv:.1f} d')
ax1.axvline(P_ast, color='C1', ls=':', label=f'astro-only P = {P_ast:.1f} d')
ax1.axvline(truth.P_days, color='C3', label=f'truth = {truth.P_days:.0f} d')
ax1.set_xlabel('joint posterior P (days)')
ax1.set_ylabel('density')
ax1.set_title('Period consistency')
ax1.legend(fontsize=8)

# CONSISTENCY check 2: projected semi-major axes (primary vs photocenter).
ax2.hist(a1_sini_au, bins=40, color='C0', alpha=0.5, density=True,
         label='a1 sin i (RV, primary)')
ax2.hist(a_phot_sini_au, bins=40, color='C1', alpha=0.5, density=True,
         label='a_phot sin i (astrometry, photocenter)')
ax2.set_xlabel('projected semi-major axis (AU)')
ax2.set_ylabel('density')
ax2.set_title('Projected-axis consistency')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Under beta = 0 the two projected axes are EQUAL (the photocentre IS the
# primary's reflex orbit) -- there is no mass factor between them. What
# the difference measures is the deficit D = 1 - a_phot sin i / a_spec,
# exactly D = beta/(B(1+beta)) ~ beta/B for beta << 1. Formed PER DRAW so
# the two channels' shared-shape correlation cancels.
D_draw = 1.0 - a_phot_sini_au / a1_sini_au
D_draw = D_draw[np.isfinite(D_draw)]
d_lo, d_med, d_hi = np.percentile(D_draw, [16, 50, 84])
half_width = 0.5 * (d_hi - d_lo)
# Self-scaled rule (the library's own, in compose_joint_seed): flag when
# |median| exceeds 3x the half-width. No absolute |D| < 0.1 cut -- 0.1
# means nothing until it is compared with THIS fit's own width.
deviates = abs(d_med) > 3.0 * half_width
print(f'deficit D = {d_med:+.4f}  [{d_lo:+.4f}, {d_hi:+.4f}]  (68% CI)')
print('  =>', 'DEVIATES -- check' if deviates else
      'consistent with 0: CONSISTENT WITH a dark companion, not proof')
print("  D is measured UNDER beta = 0 and inherits this fit's "
      'conditioning (zero jitter, flat shape prior); the luminous / SB2 /'
      '\n  blend / triple alternatives are listed in '
      'docs/model_and_likelihoods.md, section 7.')

In [ ]:
# A look at the RV data: phase fold on the joint posterior + residuals.
def rv_curve(P_days, e, tau, K, omega, gamma, times):
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    k_per_msun = semi_amplitude_kms(mass_msun=1.0, period_yr=P_yr, ecc=e,
                                    M_total_msun=M_TOTAL_MSUN)
    mass_msun = K / k_per_msun
    return rv_model(times, period_yr=P_yr, ecc=e, omega_rad=omega, tau=tau,
                    mass_msun=mass_msun, M_msun=M_TOTAL_MSUN,
                    offset_kms=gamma, epoch_ref_mjd=EPOCH_REF)


P_med = float(pct['P (d)'][1])
i_med = int(np.argmin(np.abs(Ps - P_med)))
P_s, e_s, tau_s = flat[i_med]
_, K_s, om_s, g_s = solve_rv(P_s, e_s, tau_s)


def phase_of(times, P_days, tau):
    tp = tau * P_days + EPOCH_REF
    return ((times - tp) / P_days) % 1.0


phase = phase_of(t_rv, P_s, tau_s)
ph_grid = np.linspace(0.0, 1.0, 500)
t_grid = ph_grid * P_s + (tau_s * P_s + EPOCH_REF)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
idx = rng.choice(flat.shape[0], size=80, replace=False)
for j in idx:
    Pj, ej, tauj = flat[j]
    _, Kj, omj, gj = solve_rv(Pj, ej, tauj)
    ax1.plot(ph_grid, rv_curve(Pj, ej, tauj, Kj, omj, gj, t_grid),
             color='C0', alpha=0.05, lw=1)
ax1.errorbar(phase, v, yerr=verr, fmt='o', color='k', ms=4, capsize=2, zorder=5)
ax1.set_ylabel('radial velocity (km/s)')
ax1.set_title('Phase-folded RV with joint posterior curves')

v_model = rv_curve(P_s, e_s, tau_s, K_s, om_s, g_s, t_rv)
resid_rv = v - v_model
ax2.axhline(0.0, color='C3', lw=1)
ax2.errorbar(phase, resid_rv, yerr=verr, fmt='o', color='k', ms=4, capsize=2)
ax2.set_xlabel('orbital phase')
ax2.set_ylabel('O - C (km/s)')
plt.show()
print('RV residual RMS: %.3f km/s' % np.sqrt(np.mean(resid_rv ** 2)))

In [ ]:
# A look at the astrometry: sky-plane photocenter orbit at the same
# joint posterior sample, with along-scan +/-1 sigma bars.
def plot_skyplane(P_days, e, tau, beta, ax):
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    A_ti, B_ti, F_ti, G_ti = beta[0], beta[1], beta[2], beta[3]
    ra_off, dec_off, pmra_s, pmdec_s, plx_s = beta[4:9]
    disk_x, disk_y = np.cos(2 * np.pi * tau), np.sin(2 * np.pi * tau)
    tp = tp_from_disk_angle(disk_x, disk_y, period_days=P_days,
                            epoch_ref_mjd=EPOCH_REF)

    t_curve = np.linspace(0.0, P_days, 400) + tp
    x_c, y_c = kepler_xy_orbit(t_curve, period_yr=P_yr, ecc=e, tp_mjd=tp)
    ax.plot(B_ti * x_c + G_ti * y_c, A_ti * x_c + F_ti * y_c, '-', lw=1.8,
            color='C0', label='TI orbit')

    X = ti_design_matrix(t_ast, psi, pf, f_per_day=1.0 / P_days, ecc=e,
                         tau=tau, epoch_ref_mjd=EPOCH_REF)
    model_5p = (ra_off * X[:, 4] + dec_off * X[:, 5] + pmra_s * X[:, 6]
                + pmdec_s * X[:, 7] + plx_s * X[:, 8])
    w_orbit = d_obs - model_5p  # along-scan orbit-only residual [mas]

    x_o, y_o = kepler_xy_orbit(t_ast, period_yr=P_yr, ecc=e, tp_mjd=tp)
    dRA_m, dDec_m = B_ti * x_o + G_ti * y_o, A_ti * x_o + F_ti * y_o
    sin_psi, cos_psi = np.sin(psi), np.cos(psi)
    # AC (across-scan) is model-imputed: Gaia measures along-scan only.
    AC_m = dRA_m * cos_psi - dDec_m * sin_psi
    data_dRA = w_orbit * sin_psi + AC_m * cos_psi
    data_dDec = w_orbit * cos_psi - AC_m * sin_psi

    seg0 = np.column_stack((data_dRA - sigma * sin_psi, data_dDec - sigma * cos_psi))
    seg1 = np.column_stack((data_dRA + sigma * sin_psi, data_dDec + sigma * cos_psi))
    ax.add_collection(LineCollection(np.stack((seg0, seg1), axis=1),
                                     linewidths=0.8, alpha=0.5, zorder=3,
                                     label='AL +/-1 sigma'))
    ax.scatter(data_dRA, data_dDec, s=14, marker='o', facecolor='white',
               edgecolor='black', lw=0.9, zorder=5, label='AL transits')
    ax.scatter(0.0, 0.0, marker='+', color='k', s=70, zorder=4, label='barycenter')
    ax.set_xlabel(r'$\Delta\alpha^{*}$ [mas]')
    ax.set_ylabel(r'$\Delta\delta$ [mas]')
    ax.set_aspect('equal'); ax.invert_xaxis(); ax.grid(alpha=0.3)
    ax.legend(loc='best', fontsize=8)


fig, ax = plt.subplots(figsize=(6, 6))
beta_s = solve_ti(P_s, e_s, tau_s).beta
plot_skyplane(P_s, e_s, tau_s, beta_s, ax)
ax.set_title('Joint photocenter orbit (across-scan is model-imputed)')
plt.show()

model_ast, _ = astro_along_scan_at(P_s, e_s, tau_s)
resid_ast = d_obs - model_ast
print('astrometry residual RMS: %.4f mas' % np.sqrt(np.mean(resid_ast ** 2)))

## 8. The second rung: share the angles, not just the shape

Everything above ties the two channels together through **one thing**: the
shape `(P, e, tau)`. Each channel then solves its own amplitudes -- the RV
side its `(gamma, K cos omega, K sin omega)`, the astrometric side its four
Thiele-Innes constants plus five nuisances -- and §7 *checks afterwards* that
those separately-solved amplitudes describe the same orbit. That is the
minimal joint fit: cheap, three sampled dimensions, and honest about what it
assumes.

The next rung makes "same orbit" the **model** rather than the check. Sample
six non-linear parameters, `(P, e, tau, omega, Omega, cos i)` -- the shape
*and* the three angles -- and at each step solve what is *still* linear once
the geometry is fixed:

- the RV curve is linear in **two** amplitudes, `(gamma, K)`, because `omega`
  is no longer the RV solve's to choose (`reduced_rv_design`, 2 columns);
- the along-scan model is linear in **six**: the photocentre scale `a_phot`
  and the five single-star nuisances `(ra_offset, dec_offset, pmra, pmdec,
  plx)`, because with `(omega, Omega, i)` fixed the four Thiele-Innes
  constants are all `a_phot` times a known number
  (`reduced_astro_design`, 6 columns).

The joint likelihood is, as before, the **sum of the two closed-form marginal
evidences** -- the same `linear_solve_rv` and `linear_solve_ti`, handed
narrower design matrices. Nothing new in the orbit mathematics; what changed
is what the two channels are *allowed* to disagree about. `omega` is now one
number, so the optional consistency penalty of §5b has nothing left to do.

Two things to hold in mind when reading the result.

- **The sign gauge is still there.** `(a_phot, omega, Omega)` and
  `(-a_phot, omega + pi, Omega + pi)` are the *same* orbit, so the sampler
  finds both copies. We fold `Omega` into `[0, pi)` with the compensating
  shift of `omega` when summarising -- the same fold `ti_to_kepler` applies.
- **The mirror is genuinely two geometries.** `(i, omega, Omega)` and
  `(pi - i, ...)` predict the same along-scan track. The seed decides which
  mode the chains start in; on this data the RVs break the tie.

We seed from rung 1: the median shape from §6, `omega` from the RV solve,
`Omega` and `i` from the Thiele-Innes inversion in §7. A seed is
initialisation, never a prior.

In [ ]:
from orblet.design.columns import reduced_rv_design, reduced_astro_design
from orblet.sampling import run_emcee_chains, rhat_per_param
from orblet.chain_stats import chain_circular_summary, chain_quantiles
from orblet.priors import UniformPrior


def joint_marginal_rung2(theta2):
    # Sum of the two marginal evidences at a FIXED six-parameter geometry.
    P_days, e, tau, omega, Omega, cos_i = theta2
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    X_rv = reduced_rv_design(t_rv, P_yr=P_yr, e=e, omega=omega, tau=tau, epoch_ref_mjd=EPOCH_REF)
    X_al = reduced_astro_design(t_ast, psi, pf, P_yr=P_yr, e=e, omega=omega, Omega=Omega,
                                cos_i=cos_i, tau=tau, epoch_ref_mjd=EPOCH_REF)
    sol_rv = linear_solve_rv(v, verr, X_rv)
    sol_al = linear_solve_ti(d_obs, sigma, X_al)
    return float(sol_rv.logL_marginal + sol_al.logL_marginal), sol_rv, sol_al


PRIORS2 = [
    UniformPrior(P_LO, P_HI),           # P (days), the same box as §5
    UniformPrior(0.0, 0.9),             # e
    UniformPrior(0.0, 1.0),             # tau
    UniformPrior(0.0, 2.0 * np.pi),     # omega (rad)
    UniformPrior(0.0, 2.0 * np.pi),     # Omega (rad)
    UniformPrior(-1.0, 1.0),            # cos i  (isotropic orientations)
]
LABELS2 = ['P (d)', 'e', 'tau', 'omega (rad)', 'Omega (rad)', 'cos i']


def log_post2(theta2):
    lp = sum(p.logpdf(x) for p, x in zip(PRIORS2, theta2))
    if not np.isfinite(lp):
        return -np.inf
    try:
        ll = joint_marginal_rung2(theta2)[0]
    except Exception:
        return -np.inf
    return lp + ll if np.isfinite(ll) else -np.inf


# The seed, from rung 1: median shape, omega from the RV solve, (Omega, i) from
# the Thiele-Innes inversion of §7.
P_1, e_1, tau_1 = np.median(flat, axis=0)
seed2 = np.array([P_1, e_1, tau_1,
                  chain_circular_summary(omegas)['circmean_rad'],
                  chain_circular_summary(kep['Omega_rad'])['circmean_rad'],
                  np.cos(np.median(kep['inc_rad']))])
print('rung-2 seed (P, e, tau, omega, Omega, cos i):', np.round(seed2, 3))
print('log posterior at the seed:', round(log_post2(seed2), 2))

In [ ]:
N_CHAINS2, N_WALKERS2, N_DIM2, N_ITER2 = 3, 24, 6, 1200
SCATTER2 = np.array([0.5, 0.02, 0.01, 0.05, 0.05, 0.03])


def draw_init2(rng_):
    x = seed2 + SCATTER2 * rng_.standard_normal(N_DIM2)
    x[1] = np.clip(x[1], 0.0, 0.89)
    x[2] = x[2] % 1.0
    x[3] = x[3] % (2.0 * np.pi)
    x[4] = x[4] % (2.0 * np.pi)
    x[5] = np.clip(x[5], -0.99, 0.99)
    return x


run2 = run_emcee_chains(log_post2, n_chains=N_CHAINS2, n_walkers=N_WALKERS2, n_dim=N_DIM2,
                        iterations=N_ITER2, seed=7, draw_init_fn=draw_init2, engine_name='joint-rung2')
print('acceptance per chain:', np.round(run2.per_chain_acceptance, 3))

BURN2 = N_ITER2 // 2
stacked2 = np.stack([s[BURN2:].reshape(-1, N_DIM2) for s in run2.per_chain_samples])
half2 = stacked2.shape[1] // 2
split2 = np.concatenate([stacked2[:, :half2], stacked2[:, half2:2 * half2]])
for name, rh in zip(LABELS2, rhat_per_param(split2)):
    print(f'  split R-hat  {name:12s} {rh:.3f}')
flat2 = stacked2.reshape(-1, N_DIM2)

In [ ]:
# Fold the sign gauge: Omega into [0, pi), omega shifted by pi to compensate.
flat2f = flat2.copy()
wrap = flat2f[:, 4] >= np.pi
flat2f[wrap, 4] -= np.pi
flat2f[wrap, 3] = (flat2f[wrap, 3] + np.pi) % (2.0 * np.pi)

# The amplitudes, drawn from their conditional posterior per sample (as §7 does),
# so K, gamma and a_phot carry their proper uncertainty.
K2, g2, a2 = [], [], []
for th in flat2[rng.choice(flat2.shape[0], size=2000, replace=False)]:
    _, s_rv, s_al = joint_marginal_rung2(th)
    b_rv = rng.multivariate_normal(s_rv.beta, s_rv.cov)
    b_al = rng.multivariate_normal(s_al.beta, s_al.cov)
    g2.append(b_rv[0]); K2.append(b_rv[1]); a2.append(abs(b_al[0]))
K2, g2, a2 = map(np.asarray, (K2, g2, a2))

Omega_true = truth.Omega_rad % np.pi
omega_true = (truth.omega_rad + (np.pi if truth.Omega_rad % (2 * np.pi) >= np.pi else 0.0)) % (2 * np.pi)
rows = [
    ('P (d)',        truth.P_days,      flat2f[:, 0], np.median(flat[:, 0])),
    ('e',            truth.e,           flat2f[:, 1], np.median(flat[:, 1])),
    ('tau',          ((truth.tp_mjd - EPOCH_REF) / truth.P_days) % 1.0, flat2f[:, 2], np.median(flat[:, 2])),
    ('omega (rad)',  omega_true,        flat2f[:, 3], chain_circular_summary(omegas)['circmean_rad']),
    ('Omega (rad)',  Omega_true,        flat2f[:, 4], chain_circular_summary(kep['Omega_rad'] % np.pi)['circmean_rad']),
    ('i (deg)',      np.degrees(truth.i_rad), np.degrees(np.arccos(flat2f[:, 5])), np.median(i_deg)),
    ('a_phot (mas)', truth.a_phot_mas,  a2,         np.median(a_phot_mas)),
    ('K (km/s)',     truth.K1_kms,      K2,         np.median(Ks)),
    ('gamma (km/s)', truth.gamma_kms,   g2,         np.median(gammas)),
]
print(f'{"parameter":13s} {"truth":>9s} {"rung 2":>9s} {"16%":>8s} {"84%":>8s} {"rung 1":>9s}')
for name, tv, s, r1 in rows:
    lo, med, hi = np.percentile(s, [16, 50, 84])
    tv_s = f'{tv:9.3f}' if tv is not None else '        -'
    print(f'{name:13s} {tv_s} {med:9.3f} {lo:8.3f} {hi:8.3f} {r1:9.3f}')

Read the table across: **rung 2 reproduces rung 1** on everything the two
have in common -- it must, they are the same likelihood at the same data --
and adds what rung 1 could only *check*: one `omega`, one `Omega`, one
inclination, with their own posteriors and correlations, and a photocentre
scale `a_phot` with a proper uncertainty instead of a value inherited from
four separately-solved constants. The inclination is now a sampled quantity;
compare its interval with the spread of the rung-1 `i` in §7, which came from
pushing draws of `(A, B, F, G)` through the inversion.

This six-dimensional posterior is the seed for the third rung, where the
amplitudes are sampled too and nothing is solved in closed form -- the same
step `fit_rv_orbit_all_parameters.ipynb` takes for the RV channel alone.

## 9. The third rung: sample everything

Rung 2 still solves eight amplitudes in closed form at every step. The last
rung stops solving anything: all **fourteen** parameters are walked by the
sampler -- the six-parameter geometry, the six astrometric amplitudes
`(a_phot, ra_offset, dec_offset, pmra, pmdec, plx)` and the two RV amplitudes
`(gamma, K)`. Nothing is marginalised analytically; the likelihood is the
plain Gaussian of the residuals in each channel.

Notice what does **not** change. The forward models are the *same two design
matrices* rung 2 built -- `reduced_astro_design` and `reduced_rv_design` --
multiplied by a sampled amplitude vector instead of a solved one. Between the
rungs the physics is identical; only the division of labour between algebra
and sampler moves.

Why climb this far? Three things the closed-form rungs cannot give you:

- **priors on the amplitudes** -- a positivity prior on `a_phot`, say, which
  removes the sign gauge outright (with `a_phot > 0` there is only one copy of
  each orbit, so no folding afterwards), or an informative prior on the
  parallax from elsewhere;
- **their correlations with the geometry**, measured rather than assumed
  Gaussian at a fixed shape;
- **non-Gaussian noise terms** -- a per-channel jitter, left out here to keep
  the count at fourteen, would go in exactly as in
  `fit_rv_orbit_all_parameters.ipynb`.

And why not start here: fourteen dimensions from a cold start is where
samplers wander. The seed is rung 2's median geometry with the amplitudes
solved there -- the ladder's whole point.

In [ ]:
from orblet.priors import LogUniformPrior

# The model at a full fourteen-parameter vector: the SAME design matrices as
# rung 2, times sampled amplitudes. Nothing is solved.
def joint_model_rung3(theta3):
    P_days, e, tau, omega, Omega, cos_i = theta3[:6]
    beta_al = theta3[6:12]        # a_phot, ra_offset, dec_offset, pmra, pmdec, plx
    beta_rv = theta3[12:14]       # gamma, K
    P_yr = P_days / DAYS_PER_KEPLER_YEAR
    X_al = reduced_astro_design(t_ast, psi, pf, P_yr=P_yr, e=e, omega=omega, Omega=Omega,
                                cos_i=cos_i, tau=tau, epoch_ref_mjd=EPOCH_REF)
    X_rv = reduced_rv_design(t_rv, P_yr=P_yr, e=e, omega=omega, tau=tau, epoch_ref_mjd=EPOCH_REF)
    return X_al @ beta_al, X_rv @ beta_rv


PRIORS3 = PRIORS2 + [
    LogUniformPrior(0.05, 50.0),        # a_phot (mas) > 0: removes the sign gauge
    UniformPrior(-50.0, 50.0),          # ra_offset (mas)
    UniformPrior(-50.0, 50.0),          # dec_offset (mas)
    UniformPrior(-100.0, 100.0),        # pmra (mas/yr)
    UniformPrior(-100.0, 100.0),        # pmdec (mas/yr)
    UniformPrior(0.01, 50.0),           # plx (mas)
    UniformPrior(-300.0, 300.0),        # gamma (km/s)
    LogUniformPrior(0.5, 200.0),        # K (km/s)
]
LABELS3 = LABELS2 + ['a_phot (mas)', 'ra_off (mas)', 'dec_off (mas)', 'pmra', 'pmdec', 'plx (mas)',
                     'gamma (km/s)', 'K (km/s)']


def log_post3(theta3):
    lp = sum(p.logpdf(x) for p, x in zip(PRIORS3, theta3))
    if not np.isfinite(lp):
        return -np.inf
    try:
        m_al, m_rv = joint_model_rung3(theta3)
    except Exception:
        return -np.inf
    ll = -0.5 * np.sum(((d_obs - m_al) / sigma) ** 2) - 0.5 * np.sum(((v - m_rv) / verr) ** 2)
    return lp + ll if np.isfinite(ll) else -np.inf


# Seed: rung 2's median geometry, amplitudes solved there. If the solved a_phot
# came out negative, apply the gauge flip so the seed sits inside a_phot > 0.
geom = np.median(flat2, axis=0)
geom[3] = chain_circular_summary(flat2[:, 3])['circmean_rad']
geom[4] = chain_circular_summary(flat2[:, 4])['circmean_rad']
_, s_rv, s_al = joint_marginal_rung2(geom)
b_al, b_rv = s_al.beta.copy(), s_rv.beta.copy()
if b_al[0] < 0:
    b_al[0] = -b_al[0]
    geom[3] = (geom[3] + np.pi) % (2 * np.pi)
    geom[4] = (geom[4] + np.pi) % (2 * np.pi)
seed3 = np.r_[geom, b_al, b_rv]
print('rung-3 seed:', np.round(seed3, 3))
print('log posterior at the seed:', round(log_post3(seed3), 2))

In [ ]:
N_CHAINS3, N_WALKERS3, N_DIM3, N_ITER3 = 3, 40, 14, 2500
SCATTER3 = np.r_[SCATTER2, 0.05, 0.2, 0.2, 0.2, 0.2, 0.05, 0.5, 1.0]


def draw_init3(rng_):
    x = seed3 + SCATTER3 * rng_.standard_normal(N_DIM3)
    x[1] = np.clip(x[1], 0.0, 0.89); x[2] = x[2] % 1.0
    x[3] = x[3] % (2 * np.pi); x[4] = x[4] % (2 * np.pi); x[5] = np.clip(x[5], -0.99, 0.99)
    x[6] = max(x[6], 0.06); x[11] = max(x[11], 0.02); x[13] = max(x[13], 0.6)
    return x


run3 = run_emcee_chains(log_post3, n_chains=N_CHAINS3, n_walkers=N_WALKERS3, n_dim=N_DIM3,
                        iterations=N_ITER3, seed=11, draw_init_fn=draw_init3, engine_name='joint-rung3')
print('acceptance per chain:', np.round(run3.per_chain_acceptance, 3))

BURN3 = N_ITER3 // 2
stacked3 = np.stack([s[BURN3:].reshape(-1, N_DIM3) for s in run3.per_chain_samples])
half3 = stacked3.shape[1] // 2
split3 = np.concatenate([stacked3[:, :half3], stacked3[:, half3:2 * half3]])
rh3 = rhat_per_param(split3)
for name, rh in zip(LABELS3, rh3):
    print(f'  split R-hat  {name:14s} {rh:.3f}')
flat3 = stacked3.reshape(-1, N_DIM3)

In [ ]:
# No gauge fold needed: a_phot > 0 by prior leaves one copy of each orbit.
truth3 = {
    'P (d)': truth.P_days, 'e': truth.e, 'tau': ((truth.tp_mjd - EPOCH_REF) / truth.P_days) % 1.0,
    'omega (rad)': truth.omega_rad % (2 * np.pi), 'Omega (rad)': truth.Omega_rad % (2 * np.pi),
    'cos i': np.cos(truth.i_rad), 'a_phot (mas)': truth.a_phot_mas,
    'ra_off (mas)': 0.0, 'dec_off (mas)': 0.0, 'pmra': truth.pmra_masyr, 'pmdec': truth.pmdec_masyr,
    'plx (mas)': truth.parallax_mas, 'gamma (km/s)': truth.gamma_kms, 'K (km/s)': truth.K1_kms,
}
rung2_ref = {'P (d)': np.median(flat2f[:, 0]), 'e': np.median(flat2f[:, 1]), 'tau': np.median(flat2f[:, 2]),
             'omega (rad)': np.median(flat2f[:, 3]), 'Omega (rad)': np.median(flat2f[:, 4]),
             'cos i': np.median(flat2f[:, 5]), 'a_phot (mas)': np.median(a2),
             'gamma (km/s)': np.median(g2), 'K (km/s)': np.median(K2)}
print(f'{"parameter":14s} {"truth":>9s} {"rung 3":>9s} {"16%":>8s} {"84%":>8s} {"rung 2":>9s}')
for j, name in enumerate(LABELS3):
    col = flat3[:, j]
    if 'rad' in name:
        lo, hi = np.percentile(col, [16, 84]); med = chain_circular_summary(col)['circmean_rad']
    else:
        lo, med, hi = np.percentile(col, [16, 50, 84])
    r2 = rung2_ref.get(name)
    print(f'{name:14s} {truth3[name]:9.3f} {med:9.3f} {lo:8.3f} {hi:8.3f} '
          + (f'{r2:9.3f}' if r2 is not None else '        -'))

Three rungs, one table each, the same orbit down every column: that is the
point of the ladder. Rung 3 recovers the geometry rung 2 found and the
amplitudes rung 2 solved, now with posteriors of their own and -- because
`a_phot > 0` was imposed as a prior -- no gauge fold afterwards. The parallax
and proper motion appear as fitted quantities for the first time; on this
synthetic data they are recovered to their injected values, which is the check
that the reduced design's nuisance columns and the simulator agree.

What the fourteen-dimensional fit costs is what it always costs: many more
likelihood calls for the same answer, and a real dependence on starting in the
right basin. Read the acceptance fractions and the split R-hat before the
table, every time.

## Appendix (optional): a *candidate* companion mass

> **Fenced, not the headline.** Everything above is a geometry + mass-
> function measurement.  This appendix takes one extra, clearly-labelled
> step.

The radial velocities give the SB1 **mass function** `fm_spec_msun`.  On their own,
RV data fix `sin i = 1` and so deliver only a *minimum* companion mass.
But the joint fit also has the astrometry, which **measures the
inclination** `i` (the photocenter track is two-dimensional).  Feeding the
*measured* `sin i` into the mass function turns the minimum mass into a
**candidate** companion mass.

We reuse the audited `companion_mass_from_rv_posterior` adapter: it takes a
chain dict carrying `chains['fm_spec_msun']` (the mass-function draws), an assumed
primary-mass prior, and a `sin_i` model.  We pass the joint posterior's
**measured** `sin i` per draw.  We compute `fm_spec_msun` from the audited
mass-function relation
`fm = K^3 P (1-e^2)^{3/2} / (2 pi G)` (the exact reduction the RV engine's
chain export uses; `K_kms` in m/s, `P` in s, `fm_spec_msun` in M_sun) so no new orbit
math is introduced.

**Label: candidate companion mass; dark companion (light ratio beta = 0)
assumed; `sin i` MEASURED from the astrometry.  This is NOT a compact-object
claim** -- a full beta > 0 / luminous-companion / triple / blend assessment
is deferred (see `../../docs/model_and_likelihoods.md`, §7).

In [ ]:
# Mass function per posterior draw via the audited RV-engine reduction:
#   fm = K_star^3 * P * (1 - e^2)^{3/2} / (2 pi G)   [SI -> M_sun]
# (identical formula to orblet.rv_chain; K in m/s,
# P in seconds). No new orbit math -- just the audited mass-function recipe.
sec_per_yr = DAYS_PER_KEPLER_YEAR * 86400.0
K_star_ms = Ks * 1000.0                      # km/s -> m/s
P_sec = Ps * sec_per_yr / DAYS_PER_KEPLER_YEAR  # days -> seconds (P in days)
fm_kg = (K_star_ms ** 3) * P_sec * (1.0 - es ** 2) ** 1.5 / (2.0 * np.pi * G_SI)
fm_msun = fm_kg / MSUN_KG

# The audited adapter consumes chains['fm_spec_msun'] (1-D mass-function draws) and a
# sin_i that is a scalar, a prior-spec tuple, or any object exposing
# .sample(rng, size). We want the joint posterior's MEASURED sin i (from the
# astrometry), so we wrap the per-draw measured values in a tiny .sample
# object that hands them straight back -- the adapter then carries the
# measured inclination through unchanged (no new orbit math; just feeding
# the adapter measured sin i instead of a prior draw).
# fm_msun and sin_i are row-aligned per posterior draw -- keep them in lockstep.
# Assumed primary-mass prior (m1 ~ 0.9 +/- 0.1 M_sun here -- editable).
mass = companion_mass_from_rv_posterior(
    {'chains': {'fm_spec_msun': fm_msun}},
    primary_mass_prior=('Normal', 0.9, 0.1),
    sin_i=MeasuredSinI(sin_i),   # MEASURED |sin i| from the astrometry
    seed=0,
)
m2 = mass['mass']['m2']
q = np.percentile(m2[np.isfinite(m2)], [16, 50, 84])
print('CANDIDATE companion mass (dark beta=0, sin i MEASURED from astrometry):')
print(f'  m2 = {q[1]:.2f}  (+{q[2] - q[1]:.2f} / -{q[1] - q[0]:.2f}) M_sun')
print(f'  truth injected m2 = {truth.m2_msun:.2f} M_sun')
print('  NOT a compact-object claim: beta>0 / luminous / triple / blend '
      'assessment deferred (docs/model_and_likelihoods.md, section 7).')

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(m2[np.isfinite(m2)], bins=40, color='0.7', density=True)
ax.axvline(truth.m2_msun, color='C3', label=f'truth = {truth.m2_msun:.1f} M_sun')
ax.set_xlabel('candidate companion mass m2 (M_sun)')
ax.set_ylabel('density')
ax.set_title('Candidate m2 (sin i measured; dark beta=0; NOT a BH claim)')
ax.legend()
plt.show()

## Summary

We recovered the injected SB1 orbit jointly from **both** the radial
velocities and the Gaia along-scan astrometry (typical values from a run:
`P ~ 185 d`, `e ~ 0.45`, `K ~ 67 km/s`, `i ~ 127 deg`, `a_phot ~ 2.67 mas`
-- read the §7 table, not these), sampling only the three
**shared** shape parameters `(P, e, tau)` and solving both channels' linear
amplitudes in closed form inside the likelihood.  The joint log-likelihood
is just the **sum of the two audited per-channel likelihood atoms** at the
shared shape -- no new orbit math.

The co-validation step confirmed the joint period and eccentricity are
consistent with the independent RV-only and astrometry-only fits, and that
the photocentre deficit `D = 1 - a_phot sin i / a_spec` is consistent with
zero -- which is *consistent with* a dark companion under `beta = 0`, not
proof of one.

Because the astrometry **measures** the inclination, the optional appendix
turned the RV mass function into a **candidate** companion mass -- carefully
labelled as such (dark `beta = 0`, `sin i` measured; a full luminous /
triple / blend / `beta > 0` assessment is deferred).

**Two rungs.** The fit above -- shared shape, per-channel amplitudes -- is the
minimal joint model, and §7's co-validation is where it earns trust. §8 climbs
one rung: the angles are shared too, the Thiele-Innes constants collapse to a
single `a_phot`, and "one orbit" is the model rather than the check. Its
six-parameter posterior is what a fully non-linear joint fit should start from.
§9 climbs the last rung: all fourteen parameters sampled, the same design
matrices as rung 2 multiplied by sampled amplitudes, `a_phot > 0` as a prior in
place of the gauge fold. Same orbit, three times over.
